### Ingestión del archivo "movie.csv"
#### Este notebook es un clon del 01-Ingestion File movie de la carpeta ingestion con algunas cosas modificadas para que sea carga incremental, las cosas modificadas tienen el comentario al lado, básicamente lo que se modifica es:
1. Se agrega el widget v_file_date con el nombre de las carpetas de carga incremental del contenedor bronze-inc
2. Se modifica la creación del movie_df, usando la variable del contenedor de bronze incremental y apuntando al widget v_file_date
3. Se agrega la columna file_date
4. Se elimina el paso "Escribir datos en el datalake en formato Parquet"
5. Se cambia la forma de crear la managed table en el ultimo paso, el esquema, el modo de escritura, la particion, y además se agrega un paso previo que elimine registros en caso de duplicados

In [0]:
%run "../includes/configuration"





In [0]:
%run "../includes/common_functions"






In [0]:
dbutils.widgets.text("p_environment","")
v_environment = dbutils.widgets.get("p_environment")
















In [0]:
dbutils.widgets.text("p_file_date","2024-12-16")
v_file_date = dbutils.widgets.get("p_file_date")






### Paso 1 - Leer el archivo CSV usando "DataFrameReader" de Spark
### La documentación para los comandos de Spark la sacamos de https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/index.html

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, DoubleType, StringType, DateType








In [0]:
movie_schema = StructType(fields = [
    StructField("movieId", IntegerType(), False),
    StructField("title", StringType(), True),
    StructField("budget", DoubleType(), True),
    StructField("homePage", StringType(), True),
    StructField("overview", StringType(), True),
    StructField("popularity", DoubleType(), True),
    StructField("yearReleaseDate", IntegerType(), True),
    StructField("ReleaseDate", DateType(), True),
    StructField("revenue", DoubleType(), True),
    StructField("durationTime", IntegerType(), True),
    StructField("movieStatus", StringType(), True),
    StructField("tagline", StringType(), True),
    StructField("voteAverage", DoubleType(), True),
    StructField("voteCount", IntegerType(), True)
])

In [0]:
movie_df = spark.read.option("header",True).schema(movie_schema).csv(f"{bronze_inc_folder_path}/{v_file_date}/movie.csv")






In [0]:
movie_df.printSchema()

### Paso 2 - Seleccionar solo las columnas requeridas

In [0]:
movies_selected_df = movie_df.select("movieId","title","budget","popularity","yearReleaseDate","ReleaseDate","revenue","durationTime","voteAverage","voteCount")






In [0]:
movies_selected_df = movie_df.select(movie_df.movieId,movie_df.title,movie_df.budget,movie_df.popularity,movie_df.yearReleaseDate,movie_df.ReleaseDate,movie_df.revenue,movie_df.durationTime,movie_df.voteAverage,movie_df.voteCount)


In [0]:
movies_selected_df = movie_df.select(movie_df["movieId"],movie_df["title"],movie_df["budget"],movie_df["popularity"],movie_df["yearReleaseDate"],movie_df["ReleaseDate"],movie_df["revenue"],movie_df["durationTime"],movie_df["voteAverage"],movie_df["voteCount"])


In [0]:
from pyspark.sql.functions import col






In [0]:
movies_selected_df = movie_df.select(col("movieId"),col("title"),col("budget"),col("popularity"),col("yearReleaseDate"),col("ReleaseDate"),col("revenue"),col("durationTime"),col
                                     
                                     
                                     
                                     
                                     
                                     
                                     ("voteAverage"),col("voteCount"))

In [0]:
display(movies_selected_df)

### Paso 3 - Cambiar el nombre de las columnas según lo requerido

In [0]:
movies_renamed_of = movies_selected_df \
                        .withColumnRenamed("movieId", "movie_id") \
                        .withColumnRenamed("yearReleaseDate", "year_release_date") \
                        .withColumnRenamed("ReleaseDate", "release_date") \
                        .withColumnRenamed("durationTime", "duration_time") \
                        .withColumnRenamed("voteAverage", "vote_average") \
                        .withColumnRenamed("voteCount", "vote_count") 



                        



In [0]:
display(movies_renamed_of)

### Paso 4 - Agregar la columnas "ingestion_date","environment" y "file_date" al DataFrame

In [0]:
from pyspark.sql.functions import current_timestamp, lit











In [0]:
movies_final_df = add_ingestion_date(movies_renamed_of) \
                    .withColumn("environment", lit(v_environment)) \
                    .withColumn("file_date", lit(v_file_date))


















                                    

In [0]:
display(movies_final_df)

### Paso 5 - Escribir datos en una managed table en el contenedor silver

In [0]:
#delete_dups("movie_silver_inc","movies","file_date",f"{v_file_date}")










In [0]:
#movies_final_df.write.mode("append").partitionBy("file_date").format("delta").saveAsTable("movie_silver_inc.movies")

merge_condition = 'tgt.movie_id = src.movie_id and tgt.file_date = src.file_date'
merge_delta_lake(movies_final_df,'movie_silver_inc','movies',merge_condition,'file_date')



















In [0]:
%sql
SELECT file_date, COUNT(1) 
FROM movie_silver_inc.movies
GROUP BY file_date

In [0]:
dbutils.notebook.exit("Success")






